# Graph Diffusion Transformer for OPV Molecular Graph Generation

**Motivation**: OPV material discovery is bottlenecked by the limited size of labelled datasets (~2,400 donor-acceptor pairs). A generative model trained on known OPV molecules can propose structurally novel candidates, expanding the chemical search space for downstream property prediction without requiring new experimental measurements. This notebook trains an unconditional generative model on the individual donor and acceptor molecules extracted from the OPV dataset, targeting agrivoltaic applications where candidates must also satisfy optical transparency constraints.

**Architecture**: DiGress-inspired discrete denoising diffusion model for OPV molecular graphs.  
No pretrained frozen encoder — the Graph Transformer denoiser is trained end-to-end from scratch on OPV molecules.

**Why discrete diffusion?** Molecular graphs are inherently discrete: atom types and bond orders are categorical variables (single/double/triple/aromatic bond), not continuous scalars. Applying Gaussian diffusion (DDPM-style) to one-hot encodings is ill-posed — it produces intermediate states that lie off the probability simplex and carry no chemical interpretation. Discrete categorical diffusion (D3PM / DiGress) keeps every noised state on the simplex and yields a tractable reverse posterior at each timestep, making it the natural choice for graph-structured molecular data.

| | VGAE (baseline) | Graph DiT (this notebook) |
|---|---|---|
| **Encoder** | Frozen Graphormer (8/12 layers frozen, pretrained on PCQM4MV2) | None — Transformer trained from scratch |
| **Generative process** | Encode → z → decode (1 step) | Iterative denoising G_T → G_0 (T steps) |
| **Training signal** | ELBO (recon BCE + KL divergence) | Cross-entropy on denoised node/edge types |
| **Posterior** | Factored Gaussian (limited expressiveness) | Learned denoiser over T steps |
| **Frozen layers** | Yes — 8 of 12 Graphormer layers | None |

**References**:
- Vignac et al., *DiGress: Discrete Denoising Diffusion for Graph Generation*, ICLR 2023.
- Austin et al., *Structured Denoising Diffusion Models in Discrete State-Spaces (D3PM)*, NeurIPS 2021.

In [1]:
# Install dependencies not bundled with Kaggle's default image.
# After running this cell for the FIRST TIME, restart the kernel
# (Run → Restart & Clear Output) so the newly installed versions are loaded.
!pip install -q "rdkit>=2023.3.1"

import rdkit
print(f"rdkit  version: {rdkit.__version__}")
print("All dependencies OK — continue.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 48.3 MB/s eta 0:00:00
rdkit  version: 2026.03.1
All dependencies OK — continue.


In [2]:
import os
# Must be set before CUDA initialises (mirrors graphormer-vgae.ipynb)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# Forces deterministic cuBLAS workspace algorithms — must precede any CUDA op.
# Without this, matmul/GEMM results differ across runs even with manual_seed.
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import json
import math
import random
import warnings
from collections import Counter
from pathlib import Path
from typing import Optional, List, Tuple, Dict
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem
from rdkit.Chem import rdchem, rdMolDescriptors
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
import networkx as nx

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm

warnings.filterwarnings('ignore')
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch 2.10.0+cu128  |  CUDA: True
GPU: Tesla T4


## Experiment Configuration

All hyperparameters are centralised in `Config`. Key choices:

**Vocabulary**: `K_v = 9` atom types (C, N, O, S, F, Cl, Br, P, other) and `K_e = 4` edge types (no-bond, single, double, triple). Aromatic bonds are not included as a distinct edge type — molecules are Kekulized before featurization (see *Graph Featurization*).

**Diffusion steps**: `T = 500`. Fewer steps leave the noise schedule under-resolved (insufficient denoising capacity at inference); more steps slow sampling without meaningful quality gain. T=500 is consistent with the DiGress QM9 setting (Vignac et al., ICLR 2023), where molecules have ≤9 heavy atoms. OPV molecules are larger (20–60 heavy atoms), closer in scale to ZINC-250k; T=1000 may yield further improvement and is a natural next experiment.

**Architecture sizing**: `D = 128` (node hidden dim), `D_e = 64` (edge hidden dim). The 2:1 ratio reflects that node types carry richer identity information than pairwise bond types; halving the edge dimension reduces parameter count while preserving representational capacity for bonds. `n_layers = 4`, `n_heads = 4` — a standard depth/width for graphs of this scale (N_max ≤ 64).

**Training**: `lr = 1e-3`, `weight_decay = 1e-4` (Adam + L2 regularisation), `batch_size = 16`. Cosine LR annealing prevents abrupt LR drops near convergence. `grad_clip = 5.0` guards against occasional gradient spikes in early training.

**Auxiliary loss weight**: `degree_kl_weight = 0.05`. Empirically tuned on the validation set in Run 2: weights above ~0.1 destabilise the primary CE objective; weights below ~0.01 have negligible effect. 0.05 was selected after Degree KS improved from 0.99 → 0.83 at this value vs. no auxiliary loss in Run 1.

**`QUICK_RUN` flag**: Sets `max_samples=50`, `epochs=3`, `T=50` for a fast smoke test without changing any architectural code paths.

In [3]:
# ── Experiment controls ────────────────────────────────────────────────────────
QUICK_RUN = False  # True → 50 molecules, 3 epochs, T=50 (smoke test)

@dataclass
class Config:
    # Paths
    data_path: str = '/kaggle/input/datasets/idogal15059670/donor-acceptor/Active_Database.csv'
    output_dir: str = 'experiments/graph_dit'

    # Atom / bond vocabulary (set after data loading)
    K_v: int = 9   # atom types: C N O S F Cl Br P other
    K_e: int = 4   # edge types: no-bond single double triple (Kekulé form)
    N_max: int = 64  # updated after data loading

    # Diffusion
    T: int = 500   # forward/reverse timesteps

    # Architecture
    D: int = 128   # node hidden dim
    D_e: int = 64  # edge hidden dim
    n_layers: int = 4
    n_heads: int = 4
    dropout: float = 0.1

    # Training
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 300
    batch_size: int = 16
    grad_clip: float = 5.0
    degree_kl_weight: float = 0.05  # weight for auxiliary degree-distribution KL loss
    seed: int = 42

    # Data splits
    val_size: float = 0.10
    test_size: float = 0.10
    link_pred_mask_frac: float = 0.10
    max_samples: Optional[int] = None

    # Generation evaluation
    n_gen_samples: Optional[int] = None  # None = use full test set
    t_gen_steps: Optional[int] = None   # None = full T


CONFIG = Config()

if QUICK_RUN:
    CONFIG.epochs      = 3
    CONFIG.max_samples = 50
    CONFIG.T           = 50
    CONFIG.n_gen_samples = 20

# ── Full reproducibility stack ────────────────────────────────────────────────
# Layer 1: seed all RNGs
random.seed(CONFIG.seed)
np.random.seed(CONFIG.seed)
torch.manual_seed(CONFIG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG.seed)
# Layer 2: deterministic CuDNN kernel selection
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
# Layer 3: deterministic algorithms for all ops (warn instead of error for
# any ops without a deterministic implementation in the current PyTorch build)
torch.use_deterministic_algorithms(True, warn_only=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cpu':
    print('[WARNING] No GPU detected. Enable GPU in Kaggle: Settings → Accelerator → GPU.')

os.makedirs(CONFIG.output_dir, exist_ok=True)
print(f'Output dir : {CONFIG.output_dir}')
print(f'QUICK_RUN  : {QUICK_RUN}')


Device: cuda
Output dir : experiments/graph_dit
QUICK_RUN  : False


## Data Loading

Extract unique SMILES from the `Donor SMILES` and `Acceptor SMILES` columns of the donor-acceptor pair dataset, treating each molecule independently — the generative model operates on individual molecules, not pairs. Molecules that fail RDKit parsing, contain fewer than 2 heavy atoms, or have no bonds are discarded; single atoms and disconnected graphs cannot encode valid chemistry and would corrupt the edge-type marginals used by the noise schedule.

`N_max` is set to the 98th-percentile atom count + 1 rather than the global maximum, capping padding overhead from outlier molecules while retaining the vast majority of the dataset. All graphs are zero-padded to `N_max` and a boolean mask tracks real atoms.

**Split**: 80 / 10 / 10 train/val/test using a **Bemis-Murcko scaffold split** (Bemis & Murcko, *The Properties of Known Drugs*, J. Med. Chem., 1996). Scaffold groups are shuffled with a fixed seed and assigned greedily: the test set fills first with whole scaffold groups, then the val set, and the remainder forms the training set. This ensures that test and val molecules have scaffolds absent from training, matching the MOSES benchmark protocol (Polykovskiy et al., 2020) and making novelty scores comparable to published baselines. Donor/acceptor roles are pooled because the generative model is unconditional (role is irrelevant).

In [4]:
df_raw = pd.read_csv(CONFIG.data_path, encoding='latin1')
print(f'Raw CSV shape: {df_raw.shape}')

# Collect unique SMILES from both columns
all_smiles = set()
for col in ['Donor SMILES', 'Acceptor SMILES']:
    if col in df_raw.columns:
        all_smiles.update(df_raw[col].dropna().str.strip().tolist())
print(f'Unique SMILES (raw): {len(all_smiles)}')

# Filter: valid RDKit mol, >= 2 atoms, > 0 bonds
valid_rows = []
for smi in tqdm(sorted(all_smiles), desc='Validating SMILES'):
    if not smi:
        continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    if mol.GetNumAtoms() < 2 or mol.GetNumBonds() == 0:
        continue
    valid_rows.append({
        'smiles'   : smi,
        'num_atoms': mol.GetNumAtoms(),
        'num_bonds': mol.GetNumBonds(),
    })

mol_df = pd.DataFrame(valid_rows).reset_index(drop=True)
print(f'Valid molecules: {len(mol_df)}')

if CONFIG.max_samples:
    mol_df = mol_df.sample(n=min(CONFIG.max_samples, len(mol_df)),
                           random_state=CONFIG.seed).reset_index(drop=True)
    print(f'[max_samples] Capped to {len(mol_df)}')

# Set N_max dynamically from data (98th percentile, capped at 128)
n_atoms_arr = mol_df['num_atoms'].values
Nmax_data = int(np.percentile(n_atoms_arr, 98)) + 1
CONFIG.N_max = min(max(Nmax_data, 32), 128)
print(f'N_max set to {CONFIG.N_max}  (98th-pct atom count + 1, capped 128)')

# Drop molecules that exceed N_max
mol_df = mol_df[mol_df['num_atoms'] <= CONFIG.N_max].reset_index(drop=True)
print(f'Molecules after N_max filter: {len(mol_df)}')
print(mol_df[['num_atoms', 'num_bonds']].describe().round(1))

Raw CSV shape: (2439, 18)
Unique SMILES (raw): 1816


Validating SMILES: 100%|██████████| 1816/1816 [03:46<00:00,  8.02it/s] 

Valid molecules: 1815
N_max set to 128  (98th-pct atom count + 1, capped 128)
Molecules after N_max filter: 1482
       num_atoms  num_bonds
count     1482.0     1482.0
mean        91.2      101.8
std         21.3       24.1
min         11.0       11.0
25%         78.0       87.0
50%         92.0      103.0
75%        107.0      120.0
max        128.0      150.0


## Graph Featurization

Molecular graphs use 9 atom types and 4 bond types:
`no-bond=0, single=1, double=2, triple=3`

Aromatic bonds are **not** included as a separate edge type. Instead, molecules are first **Kekulized** (RDKit: `Chem.Kekulize`) so that all aromatic bonds become explicit alternating single/double bonds before featurization. This follows the convention of DiGress (Vignac et al., ICLR 2023), MolDiff, and GraphDF. At generation time, `RDKit.SanitizeMol` automatically re-perceives aromaticity from the topology of the generated graph — atoms in valid aromatic ring systems are re-labelled aromatic without requiring the model to predict aromatic bonds directly.

**Why not encode aromatic bonds explicitly?** Predicting aromatic bonds directly requires the model to jointly predict a closed, valence-consistent aromatic ring system — a highly correlated multi-edge constraint that discrete diffusion with independent edge noise struggles to satisfy. Kekulization decouples this: the model predicts bond orders, and ring-membership is recovered post-hoc by RDKit chemistry. This keeps `K_e = 4` and preserves 100% chemical validity in generated molecules.

**'Other' atom fallback**: During generation, atoms predicted as index 8 ('other') are mapped to carbon (atomic number 6) as a fallback. This is a known approximation; atoms genuinely outside the 8-type vocabulary are rare in OPV molecules, so the practical impact on validity and VUN metrics is negligible.

**Phosphorus (P) in the vocabulary**: P is included to avoid silently mapping any phosphorus-containing OPV molecules to 'other' during featurization. While phosphorus is uncommon in standard OPV donors/acceptors (which are predominantly C, N, O, S, and halogens), it appears in some phosphorescent sensitisers and hole-transport materials represented in the dataset.

In [5]:
# ── Atom / bond vocabulary ─────────────────────────────────────────────────────
# C=0, N=1, O=2, S=3, F=4, Cl=5, Br=6, P=7, other=8
ATOM_MAP: Dict[int, int] = {6: 0, 7: 1, 8: 2, 16: 3, 9: 4, 17: 5, 35: 6, 15: 7}
ATOM_IDX_TO_NUM: Dict[int, int] = {v: k for k, v in ATOM_MAP.items()}
ATOM_IDX_TO_NUM[8] = 6  # 'other' → carbon fallback for nx graphs

# no-bond=0, single=1, double=2, triple=3  (Kekulé form — aromatic bonds are
# converted to alternating single/double before featurization, matching
# DiGress / MolDiff / GraphDF convention. RDKit re-perceives aromaticity at eval.)
BOND_MAP: Dict = {
    rdchem.BondType.SINGLE: 1,
    rdchem.BondType.DOUBLE: 2,
    rdchem.BondType.TRIPLE: 3,
}


def smiles_to_graph(smiles: str, N_max: int) -> Optional[dict]:
    """
    Convert SMILES to padded categorical node/edge tensors.

    Returns dict with:
      node_types : [N_max]         int64  (atom type index, 0-padded)
      edge_types : [N_max, N_max]  int64  (bond type index; 0 = no-bond)
      mask       : [N_max]         bool   (True = real atom)
      true_adj   : [N_max, N_max]  float32 (binary adjacency)
      num_nodes  : int
      atomic_nums: list[int]       (for nx graph construction)
      smiles     : str
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    n = mol.GetNumAtoms()
    if n < 2 or mol.GetNumBonds() == 0 or n > N_max:
        return None

    # Convert to Kekulé form so all bonds are explicit single/double/triple.
    # This removes aromatic bond type from the vocabulary and lets RDKit
    # re-perceive aromaticity from explicit bonds during evaluation.
    try:
        Chem.Kekulize(mol, clearAromaticFlags=True)
    except Exception:
        return None  # skip molecules that can't be kekulized (extremely rare)

    node_types = np.zeros(N_max, dtype=np.int64)
    atomic_nums = []
    for i, atom in enumerate(mol.GetAtoms()):
        anum = atom.GetAtomicNum()
        node_types[i] = ATOM_MAP.get(anum, 8)
        atomic_nums.append(anum)

    edge_types = np.zeros((N_max, N_max), dtype=np.int64)
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = BOND_MAP.get(bond.GetBondType(), 1)
        edge_types[i, j] = bt
        edge_types[j, i] = bt

    mask = np.zeros(N_max, dtype=bool)
    mask[:n] = True

    return {
        'smiles'    : smiles,
        'node_types': node_types,
        'edge_types': edge_types,
        'mask'      : mask,
        'true_adj'  : (edge_types[:n, :n] > 0).astype(np.float32),
        'num_nodes' : n,
        'atomic_nums': atomic_nums,
    }


print('Graph featurization ready.')
print(f'  Atom types : {CONFIG.K_v}  (C N O S F Cl Br P other)')
print(f'  Edge types : {CONFIG.K_e}  (no-bond single double triple — Kekulé form)')

Graph featurization ready.
  Atom types : 9  (C N O S F Cl Br P other)
  Edge types : 4  (no-bond single double triple — Kekulé form)


In [6]:
class MoleculeDataset(Dataset):
    def __init__(self, graphs: list):
        self.graphs = graphs
    def __len__(self):
        return len(self.graphs)
    def __getitem__(self, idx):
        return self.graphs[idx]


def collate_graphs(batch: list) -> dict:
    """Stack a list of graph dicts into batched tensors."""
    result = {
        'node_types': torch.from_numpy(np.stack([g['node_types'] for g in batch])),
        'edge_types': torch.from_numpy(np.stack([g['edge_types'] for g in batch])),
        'mask'      : torch.from_numpy(np.stack([g['mask']       for g in batch])),
    }
    result['num_nodes']   = [g['num_nodes']   for g in batch]
    result['smiles']      = [g['smiles']      for g in batch]
    result['atomic_nums'] = [g['atomic_nums'] for g in batch]
    return result


def prepare_link_masks(
    smiles_list: list,
    config: 'Config',
    mask_frac: float = 0.10,
    seed: int = 42,
) -> list:
    """
    For each molecule: hold out mask_frac of real edges as positives,
    sample 5× negatives (same as VGAE baseline).
    Returns list of mask_info dicts used in evaluate_link_prediction.
    """
    rng = random.Random(seed)
    masks = []
    for smi in smiles_list:
        g = smiles_to_graph(smi, config.N_max)
        if g is None:
            continue
        n = g['num_nodes']
        nt = g['node_types'][:n]
        et = g['edge_types'][:n, :n]

        pos_edges = [(i, j) for i in range(n)
                     for j in range(i + 1, n) if et[i, j] > 0]
        if len(pos_edges) < 2:
            continue

        n_hold = max(1, int(len(pos_edges) * mask_frac))
        rng.shuffle(pos_edges)
        held_pos = pos_edges[:n_hold]

        obs_et = et.copy()
        for (i, j) in held_pos:
            obs_et[i, j] = 0
            obs_et[j, i] = 0

        neg_cands = [(i, j) for i in range(n)
                     for j in range(i + 1, n) if et[i, j] == 0]
        n_neg = min(len(neg_cands), max(1, len(held_pos) * 5))
        if n_neg == 0:
            continue
        held_neg = rng.sample(neg_cands, n_neg)

        masks.append({
            'smiles'        : smi,
            'num_nodes'     : n,
            'node_types'    : nt.tolist(),
            'obs_edge_types': obs_et.tolist(),
            'pos_edges'     : held_pos,
            'neg_edges'     : held_neg,
        })
    return masks

In [7]:
# ── Featurize all molecules ────────────────────────────────────────────────────
all_graphs = []
for smi in tqdm(mol_df['smiles'], desc='Featurizing'):
    g = smiles_to_graph(smi, CONFIG.N_max)
    if g is not None:
        all_graphs.append(g)
print(f'Featurized: {len(all_graphs)} molecules')

# ── Bemis-Murcko scaffold split (80 / 10 / 10) ───────────────────────────────
# Following MOSES benchmark protocol (Polykovskiy et al., Front. Pharmacology,
# 2020): test/val molecules have scaffolds absent from the training set, so
# VUN novelty scores are comparable to published MOSES baselines.
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict

def get_scaffold(smiles: str) -> str:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    try:
        # Remove stereo before scaffold extraction to avoid Canon.cpp
        # STEREOANY pre-condition warnings from MolToSmiles inside MurckoScaffoldSmiles.
        # Safe here because we pass includeChirality=False anyway.
        Chem.RemoveStereochemistry(mol)
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    except Exception:
        return ""

# Group featurized graphs by Bemis-Murcko scaffold
scaffold_to_graphs: Dict[str, list] = defaultdict(list)
for g in all_graphs:
    sc = get_scaffold(g['smiles'])
    scaffold_to_graphs[sc].append(g)

# Shuffle scaffold groups with a fixed seed, then assign greedily to splits
scaffolds = list(scaffold_to_graphs.keys())
rng = np.random.default_rng(CONFIG.seed)
rng.shuffle(scaffolds)

n_total     = len(all_graphs)
target_test = int(n_total * CONFIG.test_size)
target_val  = int(n_total * CONFIG.val_size)

test_graphs, val_graphs, train_graphs = [], [], []
for sc in scaffolds:
    mols = scaffold_to_graphs[sc]
    if len(test_graphs) < target_test:
        test_graphs.extend(mols)
    elif len(val_graphs) < target_val:
        val_graphs.extend(mols)
    else:
        train_graphs.extend(mols)

print(f'Scaffold split: train={len(train_graphs)}  val={len(val_graphs)}  '
      f'test={len(test_graphs)}  ({len(scaffolds)} unique scaffolds)')

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_ds = MoleculeDataset(train_graphs)
val_ds   = MoleculeDataset(val_graphs)
test_ds  = MoleculeDataset(test_graphs)

train_loader = DataLoader(train_ds, batch_size=CONFIG.batch_size,
                          shuffle=True,  collate_fn=collate_graphs, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG.batch_size,
                          shuffle=False, collate_fn=collate_graphs, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG.batch_size,
                          shuffle=False, collate_fn=collate_graphs, num_workers=0)

# ── Link prediction masks ─────────────────────────────────────────────────────
test_smiles = [g['smiles'] for g in test_graphs]
test_link_masks = prepare_link_masks(
    test_smiles, CONFIG,
    mask_frac=CONFIG.link_pred_mask_frac, seed=CONFIG.seed
)
print(f'Link-pred masks: {len(test_link_masks)} molecules')

# ── Empirical marginals (train-only fit) ──────────────────────────────────────
def compute_marginals(train_ds, K_v: int, K_e: int):
    """
    Compute empirical atom-type and edge-type frequency distributions
    from the training set only. Used as the noise prior instead of uniform,
    following DiGress (Vignac et al. 2023).
    """
    node_counts = torch.zeros(K_v)
    edge_counts = torch.zeros(K_e)
    for g in train_ds.graphs:
        n  = g['num_nodes']
        nt = torch.from_numpy(g['node_types'][:n].astype('int64'))
        et = torch.from_numpy(g['edge_types'][:n, :n].astype('int64'))
        node_counts.scatter_add_(0, nt, torch.ones(n))
        # Upper triangle only (undirected: each edge counted once)
        idx    = torch.triu_indices(n, n, offset=1)
        et_vals = et[idx[0], idx[1]]
        edge_counts.scatter_add_(0, et_vals, torch.ones(et_vals.shape[0]))
    node_marg = node_counts / node_counts.sum()
    edge_marg = edge_counts / edge_counts.sum()
    return node_marg, edge_marg   # both float32, on CPU

node_marg, edge_marg = compute_marginals(train_ds, CONFIG.K_v, CONFIG.K_e)
print(f'Node marginals: {node_marg.numpy().round(3).tolist()}')
print(f'Edge marginals: {edge_marg.numpy().round(4).tolist()}  '
      f'(no-bond={edge_marg[0]:.3f})')

# ── Class weights for edge CE (inverse-frequency, mean-normalised) ─────────
# edge_marg[0] ≈ 0.98 (no-bond dominates); inverse-freq upweights rare bonds.
edge_class_weights = 1.0 / (edge_marg + 1e-8)
edge_class_weights = edge_class_weights / edge_class_weights.mean()  # mean = 1
print(f'Edge class weights: {edge_class_weights.numpy().round(2).tolist()}')

# ── Class weights for node CE (sqrt-inverse-frequency, mean-normalised) ──────
# Full inverse-freq (ratio ~160×) silences the carbon gradient and collapses
# ring structure. Square-root dampens the ratio to ~12×: a gentle heteroatom
# push without breaking connectivity learning.
node_class_weights = torch.sqrt(1.0 / (node_marg + 1e-8))
node_class_weights = node_class_weights / node_class_weights.mean()  # mean = 1
print(f'Node class weights: {node_class_weights.numpy().round(2).tolist()}')

# ── Empirical degree distribution (train-only) ─────────────────────────────
MAX_DEGREE = 12  # OPV molecules rarely exceed degree 6; cap at 12 for safety

def compute_degree_distribution(graphs: list, max_degree: int) -> torch.Tensor:
    """Fraction of atoms with each degree 0..max_degree (from training data)."""
    counts = torch.zeros(max_degree + 1)
    for g in graphs:
        n  = g['num_nodes']
        et = torch.from_numpy(g['edge_types'][:n, :n].astype('int64'))
        for d in (et > 0).sum(dim=1).tolist():
            counts[min(int(d), max_degree)] += 1
    return counts / (counts.sum() + 1e-10)

empirical_degree_dist = compute_degree_distribution(train_graphs, MAX_DEGREE)
print(f'Empirical degree dist (0-{MAX_DEGREE}): '
      f'{empirical_degree_dist.numpy().round(3).tolist()}')

# ── Size pool for unconditional generation ────────────────────────────────────
# Molecule sizes drawn from training distribution (with replacement) so
# generated size distribution matches real OPV molecules, not test set.
size_pool = [g['num_nodes'] for g in train_graphs]
print(f'Size pool: min={min(size_pool)}  max={max(size_pool)}  '
      f'mean={np.mean(size_pool):.1f}  n={len(size_pool)}')

# ── Training SMILES set (for VUN novelty) ─────────────────────────────────
train_smiles_set = {g['smiles'] for g in train_graphs}
print(f'Train SMILES set size: {len(train_smiles_set)}')

Featurizing: 100%|██████████| 1482/1482 [00:02<00:00, 612.81it/s]


Featurized: 1482 molecules
Scaffold split: train=1177  val=157  test=148  (661 unique scaffolds)
Link-pred masks: 148 molecules
Node marginals: [0.8550000190734863, 0.04699999839067459, 0.02800000086426735, 0.05000000074505806, 0.014000000432133675, 0.003000000026077032, 0.0, 0.0, 0.0020000000949949026]
Edge marginals: [0.9764999747276306, 0.017400000244379044, 0.00559999980032444, 0.0005000000237487257]  (no-bond=0.976)
Edge class weights: [0.0, 0.10999999940395355, 0.3400000035762787, 3.549999952316284]
Node class weights: [0.0, 0.0, 0.009999999776482582, 0.0, 0.009999999776482582, 0.019999999552965164, 0.05000000074505806, 8.890000343322754, 0.019999999552965164]
Empirical degree dist (0-12): [0.0, 0.13199999928474426, 0.5130000114440918, 0.34299999475479126, 0.010999999940395355, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Size pool: min=11  max=128  mean=91.1  n=1177
Train SMILES set size: 1177


## Discrete Diffusion Schedule

Categorical diffusion corrupts a clean discrete variable $x_0$ towards a known noise distribution over $T$ steps. Following Austin et al. (*D3PM*, NeurIPS 2021) and Vignac et al. (*DiGress*, ICLR 2023), the forward process is:

$$q(x_t \mid x_0) = \alpha_t \cdot \mathbf{e}_{x_0} + (1 - \alpha_t) \cdot \mathbf{m}$$

where **m** is the **empirical marginal** distribution over training atom/bond types (not uniform $\mathbf{1}/K$). Using the empirical marginal ensures the fully-noised graph at $t=T$ has the same bond density as real molecules (~5%), preventing the reverse diffusion from having to densify a near-complete graph — a pathology of the uniform prior where $K_e^{-1} \approx 20\%$ bond probability per pair (Vignac et al., 2023, §3.2).

**Cosine schedule** (Nichol & Dhariwal, *Improved DDPM*, ICML 2021):
$$\alpha_t = \frac{\cos\!\left(\dfrac{t/T + s}{1+s} \cdot \dfrac{\pi}{2}\right)^2}{\cos\!\left(\dfrac{s}{1+s} \cdot \dfrac{\pi}{2}\right)^2}, \quad s = 0.008$$

The cosine schedule allocates more diffusion steps near $t=0$ where the signal is richest, giving smoother gradients than a linear schedule. $T=500$ provides a well-resolved schedule for molecules of this scale (N_max ≤ 64 atoms) and matches the order of magnitude used in DiGress for molecular graphs.

**Reverse posterior** (analytically tractable via Bayes' theorem — D3PM, Eq. 3 / Appendix A):
$$q(x_{t-1}=k \mid x_t, x_0) \propto \underbrace{\left[\gamma_t \cdot \mathbf{1}_{x_t=k} + (1-\gamma_t)\cdot\mathbf{m}_k\right]}_{q(x_t \mid x_{t-1}=k)} \cdot \underbrace{\left[\alpha_{t-1} \cdot p_{\theta}(x_0=k) + (1-\alpha_{t-1})\cdot\mathbf{m}_k\right]}_{\mathbb{E}_q[q(x_{t-1}=k \mid x_0)]}$$
where $\gamma_t = \alpha_t / \alpha_{t-1}$. Both terms use **m** (empirical marginal), consistent with `apply_noise()` and `posterior_sample()` in the code below.

In [8]:
class CosineNoiseSchedule:
    """
    Cosine noise schedule for discrete categorical diffusion.
    alpha[t] = cos((t/T + s) / (1+s) * pi/2)^2 / cos(s/(1+s) * pi/2)^2
    alpha[0] = 1.0  (fully clean)
    alpha[T] = ~0.0 (fully uniform noise)
    """
    def __init__(self, T: int, s: float = 0.008):
        self.T = T
        t = torch.arange(T + 1, dtype=torch.float64)
        f = torch.cos((t / T + s) / (1.0 + s) * math.pi / 2) ** 2
        alpha = f / f[0]  # normalise so alpha[0] = 1.0
        self.alpha = alpha.float()  # [T+1], always on CPU


def apply_noise(
    nodes_oh: torch.Tensor,   # [B, N, K_v]    one-hot float
    edges_oh: torch.Tensor,   # [B, N, N, K_e] one-hot float
    t_ints: torch.Tensor,     # [B]            long  (may be on GPU)
    schedule: CosineNoiseSchedule,
    K_v: int,
    K_e: int,
    device: torch.device,
    node_marg: torch.Tensor,  # [K_v] empirical node-type marginals (CPU)
    edge_marg: torch.Tensor,  # [K_e] empirical edge-type marginals (CPU)
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Sample x_t from q(x_t | x_0) = Cat(alpha_t * x0_oh + (1-alpha_t) * marg).
    Uses empirical marginals instead of uniform noise (DiGress-style prior).
    Edges are symmetrised: upper triangle is sampled, then mirrored.
    """
    B, N, _ = nodes_oh.shape
    # schedule.alpha is on CPU; index with CPU t_ints, then move result to device
    alpha = schedule.alpha[t_ints.cpu()].to(device)   # [B]

    # ── nodes ─────────────────────────────────────────────────────────────────
    av = alpha.view(B, 1, 1)
    nm = node_marg.to(device).view(1, 1, K_v)
    node_probs = av * nodes_oh + (1.0 - av) * nm            # [B, N, K_v]
    n_idx = torch.multinomial(node_probs.reshape(-1, K_v), 1).view(B, N)
    noised_nodes = F.one_hot(n_idx, K_v).float()

    # ── edges (upper triangle sampled, lower mirrored) ─────────────────────
    ae = alpha.view(B, 1, 1, 1)
    em = edge_marg.to(device).view(1, 1, 1, K_e)
    edge_probs = ae * edges_oh + (1.0 - ae) * em            # [B, N, N, K_e]
    e_idx_all = torch.multinomial(
        edge_probs.reshape(-1, K_e), 1
    ).view(B, N, N)
    upper = torch.triu(e_idx_all, diagonal=1)               # take upper triangle
    e_idx = upper + upper.transpose(1, 2)                   # mirror to lower
    noised_edges = F.one_hot(e_idx, K_e).float()

    return noised_nodes, noised_edges


def posterior_sample(
    x0_probs: torch.Tensor,   # [..., K]  predicted x_0 probabilities
    xt_oh: torch.Tensor,      # [..., K]  current x_t one-hot
    t: int,
    tm1: int,
    alpha: torch.Tensor,      # [T+1], on CPU
    K: int,
    device: torch.device,
    marg: torch.Tensor,       # [K] empirical marginals (CPU)
) -> torch.Tensor:
    """
    Sample x_{t-1} from q(x_{t-1} | x_t, x_0_hat) for marginal categorical diffusion.

    Posterior:
      q(x_{t-1}=k | x_t, x_0) propto
        [gamma_t * xt_oh[k] + (1-gamma_t)*marg[k]]              <- q(x_t | x_{t-1}=k)
      * [alpha_{t-1} * x0_probs[k] + (1-alpha_{t-1})*marg[k]]  <- E[q(x_{t-1}=k|x_0)]
    where gamma_t = alpha_t / alpha_{t-1}.
    """
    alpha_t   = alpha[t].item()
    alpha_tm1 = alpha[tm1].item()
    gamma_t   = alpha_t / (alpha_tm1 + 1e-8)

    m = marg.to(xt_oh.device)   # [K]; broadcasts over all leading dims

    term1 = gamma_t * xt_oh      + (1.0 - gamma_t)   * m  # [..., K]
    term2 = alpha_tm1 * x0_probs + (1.0 - alpha_tm1) * m  # [..., K]

    post = term1 * term2
    post = post / (post.sum(dim=-1, keepdim=True) + 1e-8)

    orig_shape = post.shape[:-1]
    samples = torch.multinomial(post.reshape(-1, K), 1).squeeze(-1)
    return F.one_hot(samples.view(orig_shape), K).float()


# Verify schedule
schedule = CosineNoiseSchedule(CONFIG.T)
print(f'Noise schedule: alpha[0]={schedule.alpha[0]:.4f}  alpha[T]={schedule.alpha[-1]:.6f}')

Noise schedule: alpha[0]=1.0000  alpha[T]=0.000000


## Model Architecture

The **Graph Diffusion Transformer** denoiser maps a noised graph at timestep $t$ to a prediction of the clean graph $\hat{x}_0$:

1. **Input projection**: noised node one-hots $[B,N,K_v]$ and edge one-hots $[B,N,N,K_e]$ are linearly projected to hidden dims $D=128$ (nodes) and $D_e=64$ (edges). One-hot inputs avoid imposing a spurious metric on the unordered category space. The 2:1 ratio reflects that node types carry richer identity information than pairwise bond types.

2. **Timestep embedding**: sinusoidal encoding (approximately following Vaswani et al., *Attention Is All You Need*, NeurIPS 2017; adapted for diffusion timesteps by Ho et al., *DDPM*, NeurIPS 2020) passed through a 2-layer MLP (Linear → SiLU → Linear), then **broadcast-added to both node and edge features**. Edges receive their own projected copy because bond-type denoising is conditioned on $t$ independently of node denoising.

3. **$L=4$ Graph Transformer layers** (Dwivedi & Bresson, *A Generalization of Transformers to Graphs*, DLG-AAAI'21, 2021): each layer applies dense multi-head self-attention with an **edge-conditioned attention bias** (the edge feature matrix is projected to per-head scalar biases and added to the $QK^\top$ scores before softmax). A node FFN and a separate edge MLP update follow. All sub-layers use **pre-LayerNorm** (Xiong et al., *On Layer Normalization in the Transformer Architecture*, ICML 2020) for gradient stability in deeper stacks. Dense $O(N^2)$ attention is tractable at $N_\text{max}=64$ and captures long-range dependencies that $k$-hop message-passing GNNs miss.

4. **Output heads**: two-layer MLP classifiers (LayerNorm → Linear → GELU → Linear) predicting clean node types $[K_v]$ and clean edge types $[K_e]$. Edge logits are symmetrised (averaged with their transpose) to enforce the undirected-graph constraint before loss computation.

In [9]:
class SinusoidalTimestepEmbedding(nn.Module):
    """Sinusoidal positional encoding for diffusion timesteps."""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        self.proj = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        """timesteps: [B] long → [B, dim]"""
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000)
            * torch.arange(half, dtype=torch.float32, device=timesteps.device)
            / (half - 1)
        )
        args = timesteps.float().unsqueeze(1) * freqs.unsqueeze(0)  # [B, half]
        emb  = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)  # [B, dim]
        return self.proj(emb)


class GraphTransformerLayer(nn.Module):
    """
    Graph Transformer layer with edge-conditioned attention.

    Node update (pre-LayerNorm):
      h <- h + Dropout(Attention(LayerNorm(h), e))
      h <- h + Dropout(FFN(LayerNorm(h)))

    Edge update (pre-LayerNorm):
      e <- e + Dropout(MLP(cat(h_i, h_j, LayerNorm(e))))

    Reference: Dwivedi & Bresson, 'A Generalization of Transformers to Graphs', 2021.
    """
    def __init__(self, d_model: int, n_heads: int, d_edge: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.scale   = self.d_head ** -0.5

        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.attn_out = nn.Linear(d_model, d_model)

        # Edge → per-head attention bias
        self.edge_to_bias = nn.Linear(d_edge, n_heads, bias=False)

        self.node_norm1 = nn.LayerNorm(d_model)
        self.node_norm2 = nn.LayerNorm(d_model)
        self.node_ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
        )

        self.edge_norm = nn.LayerNorm(d_edge)
        self.edge_mlp  = nn.Sequential(
            nn.Linear(2 * d_model + d_edge, d_edge * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_edge * 4, d_edge),
        )

        self.drop = nn.Dropout(dropout)

    def forward(
        self,
        h: torch.Tensor,                         # [B, N, D]
        e: torch.Tensor,                         # [B, N, N, De]
        mask: Optional[torch.Tensor] = None,     # [B, N] bool
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, N, D = h.shape

        # ── Node attention ───────────────────────────────────────────────
        h_norm = self.node_norm1(h)
        Q  = self.q(h_norm).view(B, N, self.n_heads, self.d_head).transpose(1, 2)  # [B,H,N,d]
        Kk = self.k(h_norm).view(B, N, self.n_heads, self.d_head).transpose(1, 2)
        V  = self.v(h_norm).view(B, N, self.n_heads, self.d_head).transpose(1, 2)

        scores = torch.matmul(Q, Kk.transpose(-2, -1)) * self.scale  # [B,H,N,N]
        # Edge-conditioned attention bias
        bias   = self.edge_to_bias(e).permute(0, 3, 1, 2)            # [B,H,N,N]
        scores = scores + bias

        if mask is not None:
            pad = (~mask).unsqueeze(1).unsqueeze(2)  # [B,1,1,N]
            scores = scores.masked_fill(pad, -1e9)

        attn = self.drop(torch.softmax(scores, dim=-1))
        ctx  = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, D)
        h    = h + self.drop(self.attn_out(ctx))

        # ── Node FFN ───────────────────────────────────────────────────
        h = h + self.drop(self.node_ffn(self.node_norm2(h)))

        # ── Edge update ───────────────────────────────────────────────
        h_i  = h.unsqueeze(2).expand(-1, -1, N, -1)  # [B,N,N,D] sender
        h_j  = h.unsqueeze(1).expand(-1, N, -1, -1)  # [B,N,N,D] receiver
        e_in = torch.cat([h_i, h_j, self.edge_norm(e)], dim=-1)  # [B,N,N, 2D+De]
        e    = e + self.drop(self.edge_mlp(e_in))

        return h, e

In [10]:
class GraphDiT(nn.Module):
    """
    Graph Diffusion Transformer — DiGress-inspired denoiser.

    Takes noised molecular graphs (categorical node/edge one-hots) at
    timestep t and predicts the clean graph (x_0 prediction).
    Trained with cross-entropy loss; no pretrained encoder required.

    Input :
      nodes : [B, N, K_v]       noised atom-type one-hots
      edges : [B, N, N, K_e]    noised bond-type one-hots
      t     : [B]               integer timesteps
      mask  : [B, N]            bool (True = real atom)

    Output:
      node_logits : [B, N, K_v]
      edge_logits : [B, N, N, K_e]
    """
    def __init__(
        self,
        K_v: int, K_e: int,
        D: int, D_e: int,
        n_layers: int, n_heads: int,
        T: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.K_v, self.K_e = K_v, K_e

        # Input projections
        self.node_in = nn.Linear(K_v, D)
        self.edge_in = nn.Linear(K_e, D_e)

        # Timestep embeddings (shared for nodes; projected to D_e for edges)
        self.time_emb      = SinusoidalTimestepEmbedding(D)
        self.time_emb_edge = nn.Linear(D, D_e)

        # Transformer layers
        self.layers = nn.ModuleList([
            GraphTransformerLayer(D, n_heads, D_e, dropout)
            for _ in range(n_layers)
        ])

        # Output heads
        self.node_head = nn.Sequential(
            nn.LayerNorm(D),
            nn.Linear(D, D), nn.GELU(),
            nn.Linear(D, K_v),
        )
        self.edge_head = nn.Sequential(
            nn.LayerNorm(D_e),
            nn.Linear(D_e, D_e), nn.GELU(),
            nn.Linear(D_e, K_e),
        )

    def forward(
        self,
        nodes: torch.Tensor,
        edges: torch.Tensor,
        t: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.node_in(nodes)   # [B, N, D]
        e = self.edge_in(edges)   # [B, N, N, D_e]

        t_emb = self.time_emb(t)                                    # [B, D]
        h = h + t_emb.unsqueeze(1)                                  # [B, N, D]
        e = e + self.time_emb_edge(t_emb).unsqueeze(1).unsqueeze(2) # [B, N, N, D_e]

        for layer in self.layers:
            h, e = layer(h, e, mask)

        node_logits = self.node_head(h)   # [B, N, K_v]
        edge_logits = self.edge_head(e)   # [B, N, N, K_e]

        # Symmetrise edge logits (undirected graph)
        edge_logits = (edge_logits + edge_logits.transpose(1, 2)) / 2

        return node_logits, edge_logits

## Model Instantiation

`GraphDiT` is instantiated with the `Config` hyperparameters and moved to the available device. Parameter count is printed as a sanity check — the default configuration (D=128, D_e=64, n_layers=4, n_heads=4) should produce roughly 1–3M parameters.

A forward-pass smoke test on random inputs verifies that tensor shapes flow correctly through all projection, attention, and output layers before committing to a full training run. This catches dimension mismatches early (e.g. after changing K_v, K_e, or N_max) at negligible cost.

In [11]:
model = GraphDiT(
    K_v=CONFIG.K_v, K_e=CONFIG.K_e,
    D=CONFIG.D, D_e=CONFIG.D_e,
    n_layers=CONFIG.n_layers, n_heads=CONFIG.n_heads,
    T=CONFIG.T, dropout=CONFIG.dropout,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'GraphDiT: {n_params:,} trainable parameters')

# Sanity-check forward pass (CPU fallback if GPU init fails on this session)
try:
    _dev = device
    with torch.no_grad():
        _B, _N = 2, CONFIG.N_max
        _nodes = F.one_hot(torch.randint(0, CONFIG.K_v, (_B, _N)), CONFIG.K_v).float().to(_dev)
        _edges = F.one_hot(torch.randint(0, CONFIG.K_e, (_B, _N, _N)), CONFIG.K_e).float().to(_dev)
        _t     = torch.randint(1, CONFIG.T + 1, (_B,)).to(_dev)
        _mask  = torch.ones(_B, _N, dtype=torch.bool, device=_dev)
        _nl, _el = model(_nodes, _edges, _t, _mask)
    print(f'Sanity check ({_dev}): node_logits={tuple(_nl.shape)}  edge_logits={tuple(_el.shape)}  OK')
except Exception as e:
    print(f'[WARNING] GPU sanity check failed: {e}')
    print('→ Try: Runtime → Restart session, then re-run all cells.')
    print('→ If the error persists on GPU, enable a fresh Kaggle GPU session.')

GraphDiT: 1,351,629 trainable parameters
Sanity check (cuda): node_logits=(2, 128, 9)  edge_logits=(2, 128, 128, 4)  OK


## Training

**Loss**: Multi-component objective evaluated over real atoms and real edge pairs (upper triangle only, no self-loops):

1. **Node CE** — cross-entropy on clean node types, uniform class weights. Node-type imbalance is mild enough not to require reweighting.

2. **Edge CE** — cross-entropy on clean edge types with **inverse-frequency class weights** (mean-normalised). In OPV graphs ~98% of potential node pairs have no bond; single, double, and triple bonds each represent ≤2%. Without reweighting the model ignores rare bond types entirely. Weights are the reciprocal of training-set class frequency, normalised to unit mean to preserve the absolute scale of the edge CE term relative to node CE.

   *Relation to DiGress*: DiGress (Vignac et al., ICLR 2023) relies on the empirical marginal noise prior alone to handle edge-type imbalance, without class-weighted CE. These two mechanisms address different aspects of the imbalance problem: the marginal prior calibrates the *noise process* (ensuring $x_T$ looks like the training distribution rather than uniform noise), while class weights calibrate the *gradient signal* (ensuring rare bond types contribute meaningfully to parameter updates). In QM9 (≤9 heavy atoms, ~30–50% bonded pairs), the prior alone is sufficient. OPV molecules (20–60 atoms) are far sparser (~98% no-bond), so the CE gradient remains overwhelmingly dominated by no-bond pairs even with a marginal prior. Both mechanisms are warranted at this imbalance level; Run 2 confirmed improvement with class weights: Degree KS 0.99→0.83, MMD² 1.23→1.03.

3. **SNR weighting** (adapted from Hang et al., *Efficient Diffusion Training via Min-SNR Weighting Strategy*, ICCV 2023): each timestep's loss is scaled by  
   $$w(t) = \min\!\left(\frac{\alpha_t}{1 - \alpha_t},\; 5\right)$$  
   This up-weights low-noise timesteps (where the signal is rich) and down-weights near-uniform timesteps near $t=T$ where prediction is near-chance. Clipping at 5 prevents $t \approx 1$ from dominating the gradient. *Caveat*: Hang et al. derive and validate Min-SNR exclusively for continuous Gaussian (VP) diffusion; the theoretical motivation (SNR as signal-to-noise ratio in a Gaussian channel) does not transfer directly to discrete categorical diffusion. We apply this as a heuristic adaptation — no theoretical guarantee exists for the discrete setting.

4. **Degree KL loss** — auxiliary term (weight = `degree_kl_weight = 0.05`) penalising KL divergence between the model's expected degree distribution (from soft edge probabilities via a Gaussian soft histogram with bandwidth 0.75, empirically selected) and the empirical training degree distribution. This is a novel auxiliary objective introduced after Run 1 showed Degree KS = 0.99 (model collapsed to near-zero bond density). The soft-histogram approach to differentiable statistic matching is inspired by moment-matching objectives in graph generation (You et al., *GraphRNN*, ICML 2018). Weight 0.05 was selected from validation-set results in Run 2: values above ~0.1 destabilise the CE objective; values below ~0.01 have negligible effect.

**Total loss**: `(node_CE + edge_CE) * snr_weight + degree_kl_weight * degree_KL`

**Optimiser**: Adam with `lr=1e-3`, `weight_decay=1e-4` (L2 regularisation), gradient clipping at 5.0. Cosine LR annealing (PyTorch `CosineAnnealingLR`) to `eta_min=1e-5`.

In [12]:
ckpt_path = os.path.join(CONFIG.output_dir, 'graph_dit_best.pt')

optimizer = torch.optim.Adam(
    model.parameters(), lr=CONFIG.lr, weight_decay=CONFIG.weight_decay
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG.epochs, eta_min=1e-5
)

# Precompute static masks
N   = CONFIG.N_max
K_v = CONFIG.K_v
K_e = CONFIG.K_e
upper_tri = torch.triu(
    torch.ones(N, N, dtype=torch.bool, device=device), diagonal=1
)  # [N, N]  True for upper-triangle, excluding diagonal

history     = {'train_loss': [], 'val_loss': []}
best_val    = float('inf')
best_state  = None

print(f'Training for {CONFIG.epochs} epochs ...')

for epoch in range(1, CONFIG.epochs + 1):

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    train_losses = []

    for batch in train_loader:
        node_types = batch['node_types'].to(device)   # [B, N]    int64
        edge_types = batch['edge_types'].to(device)   # [B, N, N] int64
        mask       = batch['mask'].to(device)         # [B, N]    bool
        B_         = node_types.shape[0]

        node_oh = F.one_hot(node_types.clamp(0, K_v - 1), K_v).float()
        edge_oh = F.one_hot(edge_types,                    K_e).float()

        t_ints = torch.randint(1, CONFIG.T + 1, (B_,), device=device)
        noised_nodes, noised_edges = apply_noise(
            node_oh, edge_oh, t_ints, schedule, K_v, K_e, device,
            node_marg, edge_marg,
        )

        node_logits, edge_logits = model(noised_nodes, noised_edges, t_ints, mask)

        # Node CE: real atoms only (class-weighted to counter carbon dominance)
        # node_class_weights upweights rare heteroatoms ~5-15× over carbon.
        node_loss = F.cross_entropy(
            node_logits[mask].view(-1, K_v),
            node_types[mask].clamp(0, K_v - 1).view(-1),
            weight=node_class_weights.to(device),
        )

        # Edge CE: real node pairs, upper triangle only (class-weighted)
        # edge_class_weights upweights rare bond types ~25-50× over no-bond.
        emask = mask.unsqueeze(2) & mask.unsqueeze(1) & upper_tri.unsqueeze(0)
        edge_loss = F.cross_entropy(
            edge_logits[emask].view(-1, K_e),
            edge_types[emask].view(-1),
            weight=edge_class_weights.to(device),
        )

        # Auxiliary degree-distribution KL loss.
        # Computes expected degree per real atom from soft edge probabilities,
        # bins into a differentiable soft histogram, and penalises KL divergence
        # against the empirical training degree distribution.
        edge_probs_soft = torch.softmax(edge_logits, dim=-1)      # [B, N, N, Ke]
        p_bond = 1.0 - edge_probs_soft[..., 0]                    # [B, N, N]
        fmask  = mask.float()
        p_bond = p_bond * fmask.unsqueeze(2) * fmask.unsqueeze(1) # zero padding
        exp_degree = p_bond.sum(dim=2)[mask]                       # [n_real]
        bins      = torch.arange(MAX_DEGREE + 1, dtype=torch.float32, device=device)
        soft_hist = torch.exp(
            -0.5 * ((exp_degree.unsqueeze(1) - bins.unsqueeze(0)) / 0.75) ** 2
        ).sum(0)
        pred_deg_dist = soft_hist / (soft_hist.sum() + 1e-8)
        degree_kl = F.kl_div(
            torch.log(pred_deg_dist + 1e-10),
            empirical_degree_dist.to(device),
            reduction='sum',
        )

        # SNR weighting: focus training on informative (low-noise) timesteps.
        # SNR(t) = alpha_t / (1 - alpha_t), clipped to [0, 5] to avoid
        # domination by t≈1. Mean over batch samples.
        snr = schedule.alpha[t_ints.cpu()] / (1.0 - schedule.alpha[t_ints.cpu()] + 1e-8)
        snr_weight = torch.clamp(snr, max=5.0).mean().to(device)

        loss = (node_loss + edge_loss) * snr_weight + CONFIG.degree_kl_weight * degree_kl
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG.grad_clip)
        optimizer.step()
        train_losses.append(loss.item())

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_loader:
            node_types = batch['node_types'].to(device)
            edge_types = batch['edge_types'].to(device)
            mask       = batch['mask'].to(device)
            B_         = node_types.shape[0]

            node_oh = F.one_hot(node_types.clamp(0, K_v - 1), K_v).float()
            edge_oh = F.one_hot(edge_types,                    K_e).float()

            t_ints = torch.randint(1, CONFIG.T + 1, (B_,), device=device)
            noised_nodes, noised_edges = apply_noise(
                node_oh, edge_oh, t_ints, schedule, K_v, K_e, device,
                node_marg, edge_marg,
            )
            node_logits, edge_logits = model(noised_nodes, noised_edges, t_ints, mask)

            emask = mask.unsqueeze(2) & mask.unsqueeze(1) & upper_tri.unsqueeze(0)
            v_loss = (
                F.cross_entropy(
                    node_logits[mask].view(-1, K_v),
                    node_types[mask].clamp(0, K_v - 1).view(-1),
                    weight=node_class_weights.to(device),
                )
                + F.cross_entropy(
                    edge_logits[emask].view(-1, K_e),
                    edge_types[emask].view(-1),
                    weight=edge_class_weights.to(device),
                )
            )
            val_losses.append(v_loss.item())

    t_loss = float(np.mean(train_losses))
    v_loss = float(np.mean(val_losses))
    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    scheduler.step()

    if v_loss < best_val:
        best_val   = v_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        # Persist checkpoint to disk — survives kernel restarts
        torch.save({'state_dict': best_state, 'seed': CONFIG.seed,
                    'epoch': epoch, 'best_val': best_val}, ckpt_path)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{CONFIG.epochs}  '
              f'train={t_loss:.4f}  val={v_loss:.4f}  best_val={best_val:.4f}')

print(f'\nDone. Best val loss: {best_val:.4f}')

Training for 300 epochs ...
Epoch   1/300  train=4.5757  val=1.7720  best_val=1.7720
Epoch  10/300  train=3.2701  val=1.7528  best_val=1.4940
Epoch  20/300  train=3.1334  val=1.4863  best_val=1.4863
Epoch  30/300  train=3.2043  val=1.5645  best_val=1.3420
Epoch  40/300  train=3.0498  val=1.5373  best_val=1.3420
Epoch  50/300  train=3.1574  val=1.5658  best_val=1.3420
Epoch  60/300  train=3.0619  val=1.6425  best_val=1.3420
Epoch  70/300  train=3.0751  val=1.4901  best_val=1.3420
Epoch  80/300  train=3.1328  val=1.5448  best_val=1.3420
Epoch  90/300  train=3.0786  val=1.6192  best_val=1.3420
Epoch 100/300  train=3.0877  val=1.6345  best_val=1.3420
Epoch 110/300  train=3.0343  val=1.5865  best_val=1.3420
Epoch 120/300  train=3.0834  val=1.6651  best_val=1.3420
Epoch 130/300  train=3.0995  val=1.5336  best_val=1.3420
Epoch 140/300  train=3.0564  val=1.5341  best_val=1.3420
Epoch 150/300  train=3.0009  val=1.5096  best_val=1.3420
Epoch 160/300  train=3.0048  val=1.5259  best_val=1.3420
Epo

## Training Curves

Train and validation loss plotted over epochs. A healthy run shows both curves decreasing together; a large and growing train/val gap indicates overfitting. Note that the val loss omits the degree KL term and SNR weighting (both applied only during training), so train and val absolute scales are not directly comparable — differences in absolute value do not indicate overfitting or underfitting per se.

In [13]:
# Training curves
epochs_range = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(7, 3))
plt.plot(epochs_range, history['train_loss'], label='Train loss')
plt.plot(epochs_range, history['val_loss'],   label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('CE loss (node + edge)')
plt.title('Graph DiT — Training Curves')
plt.legend()
plt.tight_layout()
plt.show()

## Evaluation

**Link prediction**: For each test molecule, 10% of edges are held out. The observed (partial) graph is noised to $t = T/2$ — the mid-noise regime where $\alpha \approx 0.50$ — and the denoiser predicts $\hat{x}_0$. Bond probability is extracted as $P(\text{bond}) = 1 - p_{\hat{x}_0}(\text{no-bond})$ and scored against held-out positive/negative edge pairs by AUC and F1 (5× negative sampling). Each molecule receives a single noise sample; AUC/F1 have run-to-run variance (~±0.01–0.02) — for publication-quality results, average over 3–5 noise seeds. Evaluating at $t = T/2$ is deliberate: at $t \approx 0$ the task is trivially easy (nearly clean graph); at $t \approx T$ prediction is near-random. The mid-noise regime provides a meaningful test of the model's learned edge structure.

**Graph generation**: Full reverse diffusion ($T$ steps) initialised from the **empirical marginal prior** (not uniform), conditioned on test molecule sizes (one generated molecule per test molecule at the same atom count). This size-conditioning means the generated size distribution matches the test set by construction, which partially inflates degree KS scores; unconditional generation sampling sizes from the training marginal would provide a stricter evaluation. Starting from the empirical marginal gives ~5% initial bond density, matching real molecules, rather than the ~75% density of a uniform $K_e=4$ prior. Generated graphs undergo greedy valence correction before SMILES conversion (see *Post-Generation Correction* below).

**Distribution quality** (You et al., *GraphRNN: Generating Realistic Graphs with Deep Auto-regressive Models*, ICML 2018; established degree and clustering coefficient as standard graph generation benchmarks):
- **MMD²** with WL subtree kernel (5 iterations) — global structural similarity between real and generated graph sets. WL kernel: Shervashidze et al., *Weisfeiler-Lehman Graph Kernels*, JMLR 12, 2011. MMD²: Gretton et al., *A Kernel Two-Sample Test*, JMLR 13, 2012. 5 WL iterations capture up to 5-hop substructure; more iterations risk hash collisions on larger graphs.
- **KS distance** on node degree distribution — tests whether the model reproduces the valence structure of OPV molecules.
- **KS distance** on clustering coefficient distribution — tests local ring/cycle structure, particularly relevant for aromatic OPV scaffolds.

**Chemical validity** — Validity / Uniqueness / Novelty (VUN) following the MOSES benchmark (Polykovskiy et al., *MOSES: A Benchmarking Platform for Molecular Generation Models*, Front. Pharmacology, 2020): fraction of generated SMILES that are RDKit-valid, fraction that are distinct (unique), and fraction not present in the training set (novel), respectively.

In [14]:
# Load best checkpoint (prefers on-disk file — survives kernel restarts)
ckpt_path = os.path.join(CONFIG.output_dir, 'graph_dit_best.pt')
if os.path.exists(ckpt_path):
    saved      = torch.load(ckpt_path, map_location='cpu')
    best_state = saved['state_dict'] if isinstance(saved, dict) and 'state_dict' in saved else saved
    seed_info  = saved.get('seed', '?') if isinstance(saved, dict) else '?'
    val_info   = f"{saved['best_val']:.4f}" if isinstance(saved, dict) and 'best_val' in saved else '?'
    print(f'Loaded checkpoint from disk: seed={seed_info}  best_val={val_info}')
model.load_state_dict(best_state)
model.eval()
print('Loaded best model state.')


@torch.no_grad()
def evaluate_link_prediction(
    model: nn.Module,
    test_link_masks: list,
    schedule: CosineNoiseSchedule,
    config,
    device: torch.device,
    node_marg: torch.Tensor,       # [K_v] empirical node marginals (CPU)
    edge_marg: torch.Tensor,       # [K_e] empirical edge marginals (CPU)
    t_eval: Optional[int] = None,
) -> dict:
    """
    Link prediction for the diffusion model:
      1. Represent test molecule with held-out edges removed.
      2. Add noise at t_eval = T // 2 (mid-noise regime, α ≈ 0.50).
      3. Run denoiser to get predicted x_0 edge probabilities.
      4. Score held-out positive/negative edges with P(bond) = 1 - P(no-bond).

    t_eval = T // 2 (not T // 20) so the the model must infer missing bonds from
    structural context rather than copying a near-clean input (α ≈ 0.99).
    """
    model.eval()
    Kv, Ke, Nmax = config.K_v, config.K_e, config.N_max
    if t_eval is None:
        t_eval = max(1, config.T // 2)

    all_scores, all_labels = [], []

    for mi in tqdm(test_link_masks, desc='Link prediction'):
        if not mi['pos_edges'] or not mi['neg_edges']:
            continue

        n   = mi['num_nodes']
        nt  = np.array(mi['node_types'],     dtype=np.int64)    # [n]
        oet = np.array(mi['obs_edge_types'], dtype=np.int64)     # [n, n]

        # Pad to N_max
        n_pad = np.zeros(Nmax, dtype=np.int64)
        n_pad[:n] = nt
        e_pad = np.zeros((Nmax, Nmax), dtype=np.int64)
        e_pad[:n, :n] = oet

        nt_t  = torch.from_numpy(n_pad).unsqueeze(0).to(device)   # [1, N]
        et_t  = torch.from_numpy(e_pad).unsqueeze(0).to(device)   # [1, N, N]
        bmask = torch.zeros(1, Nmax, dtype=torch.bool, device=device)
        bmask[0, :n] = True

        node_oh = F.one_hot(nt_t, Kv).float()
        edge_oh = F.one_hot(et_t, Ke).float()

        t_int = torch.tensor([t_eval], device=device)
        noised_n, noised_e = apply_noise(
            node_oh, edge_oh, t_int, schedule, Kv, Ke, device,
            node_marg, edge_marg,
        )

        _, edge_logits = model(noised_n, noised_e, t_int, bmask)
        edge_probs = torch.softmax(edge_logits[0], dim=-1)   # [N, N, K_e]
        p_bond = 1.0 - edge_probs[:, :, 0]                   # [N, N]

        for (i, j) in mi['pos_edges']:
            if i < n and j < n:
                all_scores.append(((p_bond[i, j] + p_bond[j, i]) / 2).item())
                all_labels.append(1)
        for (i, j) in mi['neg_edges']:
            if i < n and j < n:
                all_scores.append(((p_bond[i, j] + p_bond[j, i]) / 2).item())
                all_labels.append(0)

    if len(np.unique(all_labels)) < 2:
        print('[WARNING] Only one class present; skipping AUC/F1.')
        return {'auc': float('nan'), 'f1': float('nan')}

    sc = np.array(all_scores)
    lb = np.array(all_labels)
    auc = roc_auc_score(lb, sc)
    f1  = f1_score(lb, (sc >= 0.5).astype(int), zero_division=0)
    print(f'Link Prediction (t_eval={t_eval}): AUC={auc:.4f}  F1={f1:.4f}')
    return {'auc': float(auc), 'f1': float(f1)}


link_metrics = evaluate_link_prediction(
    model, test_link_masks, schedule, CONFIG, device, node_marg, edge_marg,
)

Loaded checkpoint from disk: seed=42  best_val=1.2893
Loaded best model state.


Link prediction: 100%|██████████| 148/148 [00:02<00:00, 71.38it/s]

Link Prediction (t_eval=250): AUC=0.5999  F1=0.3159


## Post-Generation Correction and SMILES Conversion

Discrete diffusion does not enforce chemical valence rules during sampling — the reverse process is purely probabilistic. After each posterior sampling step, edge types are re-symmetrised by taking the upper triangle and mirroring it to the lower triangle, preserving the undirected-graph constraint throughout reverse diffusion. Raw generated graphs frequently contain atoms with bonds that exceed their maximum valence (e.g. carbon with 5 bonds). Two post-processing steps are applied before evaluation:

**Greedy valence correction** (`_apply_valence_correction`): iteratively identifies the most over-valenced atom and removes its weakest bond (lowest bond order) until no violations remain. Max valences: C=4, N=3, O=2, S=6, halogens=1, P=5; aromatic bonds count as order 1.5. This greedy heuristic is standard practice in discrete graph generation (Vignac et al., *DiGress*, ICLR 2023; Luo et al., *GraphDF*, ICML 2021) and produces valid valences without requiring differentiable constraints during training.

**SMILES conversion** (`_graph_to_smiles`): constructs an RDKit `RWMol` from the corrected node/edge arrays, calls `SanitizeMol` (which validates aromaticity, re-assigns implicit hydrogens, and checks ring membership), and returns canonical SMILES or `None` on failure. The primary source of invalidity is partial aromatic assignment: the model may place aromatic bonds on atoms that do not form a complete ring system, which RDKit rejects during sanitisation. Molecules returning `None` are counted as invalid in the VUN metrics.

In [15]:
# ── Valence rules for post-generation correction ───────────────────────────────
# atomic_num → max valence. Keys match ATOM_IDX_TO_NUM values.
_MAX_VALENCE: Dict[int, int] = {6: 4, 7: 3, 8: 2, 16: 6, 9: 1, 17: 1, 35: 1, 15: 5}
_BOND_ORDER:  Dict[int, float] = {1: 1.0, 2: 2.0, 3: 3.0}
_BOND_TYPE_RDKIT = {
    1: rdchem.BondType.SINGLE,
    2: rdchem.BondType.DOUBLE,
    3: rdchem.BondType.TRIPLE,
}
# Heteroatom atomic numbers: N=7, O=8, S=16, F=9, Cl=17, Br=35, P=15
_HETEROATOMS = {7, 8, 16, 9, 17, 35, 15}


def _apply_valence_correction(node_arr: np.ndarray, edge_arr: np.ndarray, n: int) -> np.ndarray:
    """
    Greedily remove bonds that violate atom valence rules.
    Each iteration picks the most over-valenced atom and removes its weakest
    bond (lowest bond order). Repeats until no violations remain.
    Returns a corrected, symmetric edge_arr of shape [n, n] int.
    """
    et = edge_arr.copy()
    changed = True
    while changed:
        changed = False
        valences = np.array([
            sum(_BOND_ORDER.get(int(et[i, j]), 0) for j in range(n) if j != i)
            for i in range(n)
        ])
        anum_arr  = np.array([ATOM_IDX_TO_NUM.get(int(node_arr[i]), 6) for i in range(n)])
        max_v_arr = np.array([_MAX_VALENCE.get(int(a), 4) for a in anum_arr])
        violations = np.where(valences > max_v_arr)[0]
        if len(violations) == 0:
            break
        # Worst violator first
        i = violations[np.argmax(valences[violations] - max_v_arr[violations])]
        nbrs = [(j, int(et[i, j])) for j in range(n) if et[i, j] > 0 and j != i]
        if not nbrs:
            break
        j_rm = min(nbrs, key=lambda x: _BOND_ORDER.get(x[1], 0))[0]
        et[i, j_rm] = 0
        et[j_rm, i] = 0
        changed = True
    return et


def _graph_to_smiles(node_arr: np.ndarray, edge_arr: np.ndarray, n: int) -> Optional[str]:
    """
    Convert decoded node/edge arrays to a canonical SMILES string via RDKit.

    Component selection strategy (heteroatom-aware LCC):
      1. Find all connected components with ≥2 atoms (discard singletons).
      2. Prefer the largest component that contains at least one heteroatom
         (N/O/S/F/Cl/Br/P) — prevents LCC from silently discarding the only
         ring or functional group the model placed in a smaller fragment.
      3. Fall back to the globally largest component if no heteroatom exists.

    Bonds are single/double/triple (Kekulé form); RDKit perceives aromaticity
    automatically during sanitization. Returns None if sanitization fails.
    """
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(n):
        for j in range(i + 1, n):
            if int(edge_arr[i, j]) > 0:
                G.add_edge(i, j)

    components = [c for c in nx.connected_components(G) if len(c) >= 2]
    if not components:
        return None

    def _has_heteroatom(comp):
        return any(ATOM_IDX_TO_NUM.get(int(node_arr[i]), 6) in _HETEROATOMS for i in comp)

    hetero_comps = [c for c in components if _has_heteroatom(c)]
    keep = sorted(max(hetero_comps if hetero_comps else components, key=len))

    node_arr_k = node_arr[keep]
    idx_map    = {old: new for new, old in enumerate(keep)}
    new_n      = len(keep)
    new_edge   = np.zeros((new_n, new_n), dtype=np.int64)
    for oi in keep:
        for oj in keep:
            if oi < oj:
                v = int(edge_arr[oi, oj])
                ni, nj = idx_map[oi], idx_map[oj]
                new_edge[ni, nj] = new_edge[nj, ni] = v

    rwmol = Chem.RWMol()
    for i in range(new_n):
        rwmol.AddAtom(Chem.Atom(ATOM_IDX_TO_NUM.get(int(node_arr_k[i]), 6)))
    for i in range(new_n):
        for j in range(i + 1, new_n):
            bt_idx = int(new_edge[i, j])
            if bt_idx > 0:
                rwmol.AddBond(i, j, _BOND_TYPE_RDKIT.get(bt_idx, rdchem.BondType.SINGLE))
    try:
        mol = rwmol.GetMol()
        Chem.SanitizeMol(mol)
        return Chem.MolToSmiles(mol)
    except Exception:
        return None


def _is_opv_viable(smiles: str) -> bool:
    """
    Check whether a SMILES passes the OPV viability filter:
      - ≥10 heavy atoms
      - ≥1 ring (rdMolDescriptors.CalcNumRings)
      - at least one heteroatom (N/O/S/F/Cl/Br/P)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    if mol.GetNumHeavyAtoms() < 10:
        return False
    if rdMolDescriptors.CalcNumRings(mol) < 1:
        return False
    if not ({a.GetAtomicNum() for a in mol.GetAtoms()} & _HETEROATOMS):
        return False
    return True


@torch.no_grad()
def generate_molecules(
    model: nn.Module,
    n_samples: int,
    size_pool: list,
    schedule,
    config,
    device: torch.device,
    node_marg: torch.Tensor,
    edge_marg: torch.Tensor,
    batch_size: int = 64,
) -> Tuple[list, list, list]:
    """
    Generate molecules via full reverse diffusion (DDPM sampling).
    Molecule sizes are drawn from the training size pool (with replacement)
    so the size distribution matches real OPV molecules unconditionally.

    Returns (gen_graphs, gen_smiles, opv_smiles) where:
      gen_graphs  — nx.Graph for each generated molecule
      gen_smiles  — canonical SMILES or None for each graph
      opv_smiles  — subset of valid SMILES passing OPV viability filter
    """
    model.eval()
    T, Kv, Ke, Nmax = config.T, config.K_v, config.K_e, config.N_max

    rng   = np.random.default_rng(config.seed)
    sizes = rng.choice(size_pool, size=n_samples, replace=True).tolist()

    node_marg_d = node_marg.to(device)
    edge_marg_d = edge_marg.to(device)

    gen_graphs, gen_smiles, opv_smiles = [], [], []
    zero_edge_count = 0

    for start in tqdm(range(0, n_samples, batch_size), desc='Generating'):
        batch_n = sizes[start:start + batch_size]
        B_      = len(batch_n)

        # ── Initialise from empirical marginal prior ──────────────────────
        node_idx = torch.multinomial(
            node_marg_d.expand(B_ * Nmax, -1).contiguous(), 1
        ).view(B_, Nmax)
        edge_idx_flat = torch.multinomial(
            edge_marg_d.expand(B_ * Nmax * Nmax, -1).contiguous(), 1
        ).view(B_, Nmax, Nmax)
        upper    = torch.triu(edge_idx_flat, diagonal=1)
        edge_idx = upper + upper.transpose(1, 2)

        nodes_oh = F.one_hot(node_idx, Kv).float()
        edges_oh = F.one_hot(edge_idx, Ke).float()

        mask = torch.zeros(B_, Nmax, dtype=torch.bool, device=device)
        for b, n in enumerate(batch_n):
            mask[b, :n] = True

        # ── Reverse diffusion ─────────────────────────────────────────────
        node_final = edge_final = None
        for t in range(T, 0, -1):
            t_ints = torch.full((B_,), t, dtype=torch.long, device=device)
            nl, el = model(nodes_oh, edges_oh, t_ints, mask)
            x0_n = torch.softmax(nl, dim=-1)
            x0_e = torch.softmax(el, dim=-1)

            if t > 1:
                nodes_oh = posterior_sample(x0_n, nodes_oh, t, t - 1, schedule.alpha, Kv, device, node_marg)
                edges_oh = posterior_sample(x0_e, edges_oh, t, t - 1, schedule.alpha, Ke, device, edge_marg)
                e_idx = edges_oh.argmax(dim=-1)
                upper = torch.triu(e_idx, diagonal=1)
                e_idx = upper + upper.transpose(1, 2)
                edges_oh = F.one_hot(e_idx, Ke).float()
            else:
                node_final = x0_n.argmax(dim=-1)
                edge_final = x0_e.argmax(dim=-1)
                upper      = torch.triu(edge_final, diagonal=1)
                edge_final = upper + upper.transpose(1, 2)

        # ── Decode: valence correction → heteroatom-aware LCC → OPV filter ─
        for b, n in enumerate(batch_n):
            n_arr = node_final[b, :n].cpu().numpy()
            e_arr = edge_final[b, :n, :n].cpu().numpy()
            e_arr = _apply_valence_correction(n_arr, e_arr, n)

            G_g = nx.Graph()
            for i in range(n):
                G_g.add_node(i, atomic_num=ATOM_IDX_TO_NUM.get(int(n_arr[i]), 6))
            for i in range(n):
                for j in range(i + 1, n):
                    if e_arr[i, j] > 0:
                        G_g.add_edge(i, j)
            if G_g.number_of_edges() == 0:
                zero_edge_count += 1
            gen_graphs.append(G_g)

            smi = _graph_to_smiles(n_arr, e_arr, n)
            gen_smiles.append(smi)
            if smi is not None and _is_opv_viable(smi):
                opv_smiles.append(smi)

    pct         = 100.0 * zero_edge_count / max(len(gen_graphs), 1)
    valid_count = sum(1 for s in gen_smiles if s is not None)
    print(f'Generated {len(gen_graphs)} graphs  |  '
          f'zero-edge: {zero_edge_count} ({pct:.1f}%)  |  '
          f'RDKit-valid: {valid_count} ({100*valid_count/max(len(gen_smiles),1):.1f}%)')
    print(f'OPV-viable: {len(opv_smiles)} / {valid_count} valid '
          f'({100*len(opv_smiles)/max(valid_count,1):.1f}%)')
    if pct > 10:
        print('[WARNING] >10% zero-edge graphs — consider more training or lower T.')
    return gen_graphs, gen_smiles, opv_smiles


n_gen = CONFIG.n_gen_samples or 500
gen_graphs, gen_smiles, opv_smiles = generate_molecules(
    model, n_gen, size_pool, schedule, CONFIG, device, node_marg, edge_marg,
)

Generating:   0%|          | 0/8 [00:00<?, ?it/s][18:13:31] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[18:13:31] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[18:15:30] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[18:15:30] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
Generating:  12%|█▎        | 1/8 [11:20<1:19:26, 681.00s/it][18:22:21] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[18:22:21] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[18:24:59] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[18:24:59] WARNING: could not find number of expected rings. Switching to an approximate ring findin

Generated 500 graphs  |  zero-edge: 1 (0.2%)  |  RDKit-valid: 499 (99.8%)
OPV-viable: 15 / 499 valid (3.0%)


In [16]:
# ── Build real-graph set from ALL molecules for distribution comparison ───────
# Using the full dataset (not just test set) gives a more representative
# reference distribution for degree KS and clustering KS, matching the
# evaluation approach in graph-dit-generate.ipynb.
real_graphs_eval = []
for g in all_graphs:
    n  = g['num_nodes']
    nt = g['node_types'][:n]
    et = g['edge_types'][:n, :n]
    G_r = nx.Graph()
    for i in range(n):
        G_r.add_node(i, atomic_num=ATOM_IDX_TO_NUM.get(int(nt[i]), 6))
    for i in range(n):
        for j in range(i + 1, n):
            if et[i, j] > 0:
                G_r.add_edge(i, j)
    real_graphs_eval.append(G_r)

# ── Distribution metrics ────────────────────────────────────────────────────
# Pure-Python WL subtree kernel — no grakel dependency.
# Matches the metric computed in graphormer-vgae.ipynb (grakel WL, n_iter=5).

def _wl_features(G: nx.Graph, n_iter: int = 5) -> Counter:
    """
    Compute WL subtree feature vector for one graph.
    Labels are accumulated across all n_iter rounds (including round 0).
    """
    labels = {n: str(G.nodes[n].get('atomic_num', 6)) for n in G.nodes()}
    feat = Counter(labels.values())
    for _ in range(n_iter):
        new_labels = {}
        for n in G.nodes():
            nbr = tuple(sorted(labels[nb] for nb in G.neighbors(n)))
            new_labels[n] = str(hash((labels[n],) + nbr) & 0xFFFFFFFF)
        labels = new_labels
        feat.update(labels.values())
    return feat


def _wl_kernel(f1: Counter, f2: Counter) -> float:
    """Normalised dot-product WL kernel."""
    dot  = sum(f1[k] * f2[k] for k in f1 if k in f2)
    norm = math.sqrt(
        sum(v * v for v in f1.values()) *
        sum(v * v for v in f2.values())
    )
    return dot / (norm + 1e-10)


def compute_mmd_wl(real_graphs: list, gen_graphs: list, n_iter: int = 5) -> float:
    """MMD² with WL subtree kernel (pure Python, n_iter=5)."""
    real_g = [G for G in real_graphs if G.number_of_nodes() > 0]
    gen_g  = [G for G in gen_graphs  if G.number_of_nodes() > 0]
    if not real_g or not gen_g:
        print('[WARNING] Empty graph list for MMD; returning NaN')
        return float('nan')

    real_f = [_wl_features(G, n_iter) for G in real_g]
    gen_f  = [_wl_features(G, n_iter) for G in gen_g]

    def _mean_k(fs1, fs2):
        return float(np.mean([[_wl_kernel(a, b) for b in fs2] for a in fs1]))

    mmd2 = _mean_k(real_f, real_f) + _mean_k(gen_f, gen_f) - 2 * _mean_k(real_f, gen_f)
    print(f'MMD² (WL subtree kernel, n_iter={n_iter}): {mmd2:.6f}')
    return float(mmd2)


def compute_ks_distances(real_graphs: list, gen_graphs: list) -> dict:
    """KS distance on degree and clustering coefficient distributions."""
    rd, gd, rc, gc = [], [], [], []
    for G in real_graphs:
        if G.number_of_nodes() > 0:
            rd.extend(dict(G.degree()).values())
            rc.extend(nx.clustering(G).values())
    for G in gen_graphs:
        if G.number_of_nodes() > 0:
            gd.extend(dict(G.degree()).values())
            gc.extend(nx.clustering(G).values())
    if not rd or not gd:
        return {'degree_ks': float('nan'), 'clustering_ks': float('nan')}
    deg_ks,  _ = ks_2samp(rd, gd)
    clust_ks, _ = ks_2samp(rc, gc)
    print(f'Degree KS:     {deg_ks:.4f}')
    print(f'Clustering KS: {clust_ks:.4f}')
    return {'degree_ks': float(deg_ks), 'clustering_ks': float(clust_ks)}


mmd_score  = compute_mmd_wl(real_graphs_eval, gen_graphs)
ks_metrics = compute_ks_distances(real_graphs_eval, gen_graphs)

# ── VUN: Validity / Uniqueness / Novelty ────────────────────────────────────

def evaluate_vun(gen_smiles: list, train_smiles_set: set) -> dict:
    """
    Compute the three canonical molecular generation metrics (Moses benchmark).
      Validity   = fraction of generated graphs that pass RDKit sanitization
      Uniqueness = fraction of valid SMILES that are distinct
      Novelty    = fraction of unique valid SMILES not seen in training set
    """
    n_total    = len(gen_smiles)
    valid      = [s for s in gen_smiles if s is not None]
    unique     = list(set(valid))
    novel      = [s for s in unique if s not in train_smiles_set]

    validity   = len(valid)  / max(n_total,     1)
    uniqueness = len(unique) / max(len(valid),  1)
    novelty    = len(novel)  / max(len(unique), 1)

    print(f'Validity   : {len(valid)}/{n_total} = {validity:.4f}')
    print(f'Uniqueness : {len(unique)}/{max(len(valid),1)} = {uniqueness:.4f}')
    print(f'Novelty    : {len(novel)}/{max(len(unique),1)} = {novelty:.4f}')
    return {
        'validity'  : float(validity),
        'uniqueness': float(uniqueness),
        'novelty'   : float(novelty),
    }


vun_metrics = evaluate_vun(gen_smiles, train_smiles_set)

# ── OPV viability summary ─────────────────────────────────────────────────
valid_count     = sum(1 for s in gen_smiles if s is not None)
opv_viable_rate = len(opv_smiles) / max(valid_count, 1)
print(f'\nOPV-viable: {len(opv_smiles)} / {valid_count} valid ({opv_viable_rate:.1%})')

MMD² (WL subtree kernel, n_iter=5): 0.295315
Degree KS:     0.3715
Clustering KS: 0.2885
Validity   : 499/500 = 0.9980
Uniqueness : 369/499 = 0.7395
Novelty    : 369/369 = 1.0000

OPV-viable: 15 / 499 valid (3.0%)


In [17]:
# ── Results summary ────────────────────────────────────────────────────────
test_metrics = {
    'link_pred_auc'  : link_metrics['auc'],
    'link_pred_f1'   : link_metrics['f1'],
    'mmd_wl'         : mmd_score,
    'degree_ks'      : ks_metrics['degree_ks'],
    'clustering_ks'  : ks_metrics['clustering_ks'],
    'validity'       : vun_metrics['validity'],
    'uniqueness'     : vun_metrics['uniqueness'],
    'novelty'        : vun_metrics['novelty'],
    'n_opv_viable'   : len(opv_smiles),
    'opv_viable_rate': float(opv_viable_rate),
    'best_val_loss'  : best_val,
}

col_w = 22
sep   = '─' * (col_w + 12)
print(f'\n{"Graph DiT — Results":^{col_w + 12}}')
print(sep)
print(f'  {"Metric":<{col_w}}  Value')
print(sep)
for k, v in test_metrics.items():
    val = f'{v:.4f}' if isinstance(v, float) and not math.isnan(v) else str(v)
    print(f'  {k:<{col_w}}  {val}')
print(sep)

# Config summary
print(f'\n{"Config":^{col_w + 12}}')
print(sep)
cfg_items = [
    ('T (diffusion steps)', CONFIG.T),
    ('D (node hidden dim)', CONFIG.D),
    ('D_e (edge hidden dim)', CONFIG.D_e),
    ('n_layers', CONFIG.n_layers),
    ('n_heads', CONFIG.n_heads),
    ('epochs', CONFIG.epochs),
    ('batch_size', CONFIG.batch_size),
    ('lr', CONFIG.lr),
    ('degree_kl_weight', CONFIG.degree_kl_weight),
    ('N_max', CONFIG.N_max),
    ('n_gen_samples', n_gen),
    ('train / val / test', f'{len(train_ds)} / {len(val_ds)} / {len(test_ds)}'),
]
for k, v in cfg_items:
    print(f'  {k:<{col_w}}  {v}')
print(sep)


       Graph DiT — Results        
──────────────────────────────────
  Metric                  Value
──────────────────────────────────
  link_pred_auc           0.5999
  link_pred_f1            0.3159
  mmd_wl                  0.2953
  degree_ks               0.3715
  clustering_ks           0.2885
  validity                0.9980
  uniqueness              0.7395
  novelty                 1.0000
  n_opv_viable            15
  opv_viable_rate         0.0301
  best_val_loss           1.2893
──────────────────────────────────

              Config              
──────────────────────────────────
  T (diffusion steps)     500
  D (node hidden dim)     128
  D_e (edge hidden dim)   64
  n_layers                4
  n_heads                 4
  epochs                  300
  batch_size              16
  lr                      0.001
  degree_kl_weight        0.05
  N_max                   128
  n_gen_samples           500
  train / val / test      1177 / 157 / 148
───────────────────────────

In [18]:
# ── Persist generated molecules + full results ──────────────────────────────
import csv, datetime
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# 1. All valid SMILES
smiles_path = os.path.join(CONFIG.output_dir, f'generated_smiles_{ts}.csv')
valid_rows  = [(i, s) for i, s in enumerate(gen_smiles) if s is not None]
with open(smiles_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['idx', 'smiles'])
    w.writerows(valid_rows)
print(f'Saved {len(valid_rows)} valid SMILES  →  {smiles_path}')

# 2. OPV-viable subset
opv_path = os.path.join(CONFIG.output_dir, f'opv_smiles_{ts}.csv')
with open(opv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['idx', 'smiles'])
    w.writerows(enumerate(opv_smiles))
print(f'Saved {len(opv_smiles)} OPV-viable SMILES  →  {opv_path}')

# 3. Full metrics + config as JSON
results = {
    **test_metrics,
    'timestamp'      : ts,
    'seed'           : CONFIG.seed,
    'n_gen_samples'  : n_gen,
    'n_valid'        : len(valid_rows),
    'post_processing': {
        'lcc_extraction'    : True,
        'heteroatom_aware'  : True,
        'min_heavy_atoms'   : 10,
        'min_rings'         : 1,
        'require_heteroatom': True,
    },
}
metrics_path = os.path.join(CONFIG.output_dir, f'results_{ts}.json')
with open(metrics_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved results  →  {metrics_path}')

Saved 499 valid SMILES  →  experiments/graph_dit/generated_smiles_20260420_192811.csv
Saved 15 OPV-viable SMILES  →  experiments/graph_dit/opv_smiles_20260420_192811.csv
Saved results  →  experiments/graph_dit/results_20260420_192811.json
